In [34]:
!pip install langchain langchain-community faiss-cpu PyPDF2 gradio sentence-transformers transformers


In [35]:
import gradio as gr
import traceback
from transformers import pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.llms import HuggingFacePipeline
from langchain.tools import Tool


def create_vectorstore(pdf_path):
    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        if not pages:
            raise Exception(" No text found. Is it scanned?")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = splitter.split_documents(pages)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        return FAISS.from_documents(chunks, embeddings)
    except Exception as e:
        raise Exception(f"PDF Error: {e}")


def create_agent(vectorstore):
    qa_pipeline = pipeline("text2text-generation", model="google/flan-t5-base")
    llm = HuggingFacePipeline(pipeline=qa_pipeline)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    return ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)


def calculator_tool(expression):
    try:
        return str(eval(expression))
    except Exception:
        return "Invalid math expression."

calculator = Tool(name="Calculator", func=calculator_tool, description="Performs simple math.")


def chat_with_agent(message, history, chain):
    try:
        if any(op in message for op in ["+", "-", "*", "/"]):
            return calculator.func(message)
        result = chain({"question": message})
        answer = result.get("answer", "").strip()
        if not answer or "could not find" in answer.lower():
            return " I couldn’t find a clear answer in the document."
        return answer
    except Exception as e:
        traceback.print_exc()
        return f" Error: {e}"


def handle_pdf_upload(pdf_file):
    if not pdf_file:
        return None, " Please upload a PDF first."
    try:
        vectorstore = create_vectorstore(pdf_file.name)
        chain = create_agent(vectorstore)
        return chain, " PDF processed successfully! You can start chatting."
    except Exception as e:
        return None, f" {e}"


with gr.Blocks(theme="soft", title="PDF Chatbot Agent") as demo:
    gr.Markdown("## PDF Chat Agent + Calculator Tool\nUpload your PDF, process it, and start chatting about its contents!")

    with gr.Row():
        pdf_file = gr.File(label=" Upload your PDF")
        process_btn = gr.Button(" Process PDF")

    status_box = gr.Textbox(label="Status", interactive=False)

    chatbot = gr.Chatbot(label="Chat Interface", height=400)
    user_input = gr.Textbox(label="Your Question", placeholder="Ask about the document or type 25 + 37 ...")

    chain_state = gr.State(None)


    def process_pdf(pdf_file):
        chain, msg = handle_pdf_upload(pdf_file)
        return chain, msg

    process_btn.click(process_pdf, inputs=pdf_file, outputs=[chain_state, status_box])


    def chat_action(user_input, history, chain):
        if not chain:
            history.append((user_input, "Please process a PDF first."))
            return history, chain
        answer = chat_with_agent(user_input, history, chain)
        history.append((user_input, answer))
        return history, chain

    user_input.submit(chat_action, inputs=[user_input, chatbot, chain_state], outputs=[chatbot, chain_state])

demo.launch(share=True)


/tmp/ipython-input-2178613502.py:91: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat Interface", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://853b424082149f6397.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
